<a href="https://colab.research.google.com/github/MarquiseRosier/pi05-run/blob/feature/marquise-transcoder-feature-inspection/notebooks/pi05_transcoder_counterfactual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Pi0.5 Transcoder Counterfactual Probe

Standalone notebook for one study: **which transcoder features respond to a
single object's appearance?**

It recolors one object at a frozen simulator state, renders the scene twice,
and pushes both observations through the policy with the same flow-matching
noise. Physics is never stepped between the renders, so any activation
difference is attributable to that object's pixels.

## Why this is separate from the main notebook

The full simulation notebook mounts Drive caches, downloads the LIBERO dataset
and runs closed-loop evals. None of that is needed here. This probe uses
`make_env`, not `make_dataset`, so there is **no dataset download**, and it
runs a handful of forward passes rather than rollouts.

Drive is mounted for one reason only: to read the transcoder checkpoint.

## Order

Run the cells top to bottom. Nothing needs to be filled in: the probe reads the
task's BDDL and perturbs its `obj_of_interest`, using another instance of the
same object type as a matched control.

1. Controls
2. Runtime check and Drive mount
3. Install
4. Clone repo
5. HF token and LIBERO assets
6. Resolve checkpoint
7. Run the probe


In [ ]:
# @title Controls

# Repo.
REPO_URL = "https://github.com/MarquiseRosier/pi05-run.git"  # @param {type:"string"}
REPO_BRANCH = "feature/marquise-transcoder-feature-inspection"  # @param {type:"string"}

# Drive is used only to read the transcoder checkpoint.
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908"  # @param {type:"string"}
TRANSCODER_DRIVE_PATH = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"  # @param {type:"string"}

# Scene.
POLICY_PATH = "lerobot/pi05_libero_finetuned"  # @param {type:"string"}
SUITE = "libero_spatial"  # @param ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
TASK_ID = 0  # @param {type:"integer"}
SEED = 1000  # @param {type:"integer"}

# Perturbation. Both default to the task's own objects: TARGET becomes the
# BDDL's first obj_of_interest, PLACEBO_TARGET another instance of the same
# object type. Leave them blank unless you want to override.
TARGET = ""  # @param {type:"string"}
PLACEBO_TARGET = ""  # @param {type:"string"}
LIST_OBJECTS_ONLY = False  # @param {type:"boolean"}
PERTURBATION = "blend"  # @param ["blend", "set", "hue"]
COLOR = "1.0,0.2,0.1"  # @param {type:"string"}
DOSE = "0.5,1.0"  # @param {type:"string"}
STATES = 2  # @param {type:"integer"}
STATE_STRIDE = 5  # @param {type:"integer"}

MIN_GPU_MEMORY_GB = 20  # @param {type:"integer"}

print("Suite:", SUITE, "| task:", TASK_ID, "| seed:", SEED)
print("Target:", repr(TARGET) or "(discovery pass)")


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())


In [ ]:
# @title Install Runtime

import os
import subprocess
from pathlib import Path

VENV = Path("/content/lerobot-venv")
PYTHON = VENV / "bin/python"
UV_BIN_DIR = Path("/content/uv-bin")
UV = str(UV_BIN_DIR / "uv")


def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, env=env, check=True)


if PYTHON.exists():
    print("venv already present; skipping install. Delete /content/lerobot-venv to force a rebuild.")
else:
    apt_packages = [
        "build-essential", "cmake", "curl", "ffmpeg", "git", "pkg-config",
        "libegl1", "libgl1", "libglib2.0-0", "libglvnd0", "libglx0", "libopengl0",
        "libosmesa6-dev", "libsm6", "libxext6", "libxrender1",
    ]
    apt_env = os.environ.copy()
    apt_env["DEBIAN_FRONTEND"] = "noninteractive"
    run(["apt-get", "update", "-qq"], env=apt_env)
    run(["apt-get", "install", "-y", "-qq", *apt_packages], env=apt_env)

    UV_BIN_DIR.mkdir(parents=True, exist_ok=True)
    run(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", "/tmp/install-uv.sh"])
    uv_env = os.environ.copy()
    uv_env["UV_INSTALL_DIR"] = str(UV_BIN_DIR)
    run(["sh", "/tmp/install-uv.sh"], env=uv_env)
    os.environ["PATH"] = f"{UV_BIN_DIR}:" + os.environ["PATH"]

    run([UV, "python", "install", "3.12"])
    run([UV, "venv", "--clear", str(VENV), "--python", "3.12"])
    # `evaluation` pulls the LIBERO env; no dataset extras are needed for this probe.
    run([
        UV, "pip", "install", "--python", str(PYTHON), "--torch-backend", "cu128",
        "lerobot[evaluation,libero,pi]", "hf-transfer", "opencv-python", "numpy",
    ])

os.environ["PATH"] = f"{VENV / 'bin'}:{UV_BIN_DIR}:" + os.environ["PATH"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
run([str(PYTHON), "-c",
     "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])


In [ ]:
# @title Clone Or Update Repo

import subprocess
from pathlib import Path

LOCAL_REPO = Path("/content/pi05-run")
if LOCAL_REPO.exists():
    subprocess.run(["git", "-C", str(LOCAL_REPO), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(LOCAL_REPO), "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(LOCAL_REPO)], check=True)
subprocess.run(["git", "-C", str(LOCAL_REPO), "log", "-1", "--oneline"], check=True)


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-c", assets_code])


In [ ]:
# @title Resolve Transcoder Checkpoint

import shutil
import time
from pathlib import Path

candidate = Path(TRANSCODER_DRIVE_PATH)
if not candidate.is_absolute():
    candidate = DRIVE_ROOT / candidate
if not candidate.exists():
    raise FileNotFoundError(
        f"Transcoder checkpoint not found: {candidate}\n"
        "Check TRANSCODER_DRIVE_PATH in Controls against the shared Drive folder."
    )

# Copy off Drive once. Reading a multi-GiB checkpoint repeatedly over the Drive
# FUSE mount is far slower than a local read, and it is the checkpoint load that
# the probe does on every run.
LOCAL_CHECKPOINT = Path("/content/checkpoints") / candidate.name
LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_CHECKPOINT.exists() and LOCAL_CHECKPOINT.stat().st_size == candidate.stat().st_size:
    print("Local copy already present:", LOCAL_CHECKPOINT)
else:
    print(f"Copying {candidate.stat().st_size / 1024**3:.2f} GiB off Drive...", flush=True)
    started = time.time()
    shutil.copy2(candidate, LOCAL_CHECKPOINT)
    print(f"  done in {time.time() - started:.0f}s", flush=True)

TRANSCODER_CHECKPOINT = str(LOCAL_CHECKPOINT)
print("Checkpoint:", TRANSCODER_CHECKPOINT)
print("Size:", f"{LOCAL_CHECKPOINT.stat().st_size / 1024**3:.2f} GiB")


## Run The Probe

Just run it. The probe reads the task's BDDL and picks its own objects:

* **target** -- the task's first `obj_of_interest`. For `libero_spatial` task 0
  that is `akita_black_bowl_1`, the bowl the prompt refers to.
* **placebo** -- another instance of the same object type, here
  `akita_black_bowl_2`. Same mesh, same colour, same size, differing only in
  position and task relevance, which makes it a tightly matched control.

Override `TARGET` / `PLACEBO_TARGET` in Controls only if you want something
else, or tick `LIST_OBJECTS_ONLY` to just inspect the scene.


In [ ]:
# @title Run Counterfactual Probe

import json
import os
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import display, HTML, Markdown, Image as IPyImage

out_dir = LOCAL_REPO / "outputs/probes/counterfactual" / time.strftime("%Y%m%d-%H%M%S", time.gmtime())
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
    "--policy-path", POLICY_PATH,
    "--output-dir", str(out_dir),
    "--suite", SUITE,
    "--task-id", str(TASK_ID),
    "--seed", str(SEED),
    "--device", "cuda",
    "--policy-dtype", "bfloat16",
]

target = TARGET.strip()
if LIST_OBJECTS_ONLY:
    cmd.append("--list-objects")
else:
    cmd += [
        "--checkpoint", TRANSCODER_CHECKPOINT,
        "--perturbation", PERTURBATION,
        "--color", COLOR,
        "--dose", DOSE,
        "--states", str(STATES),
        "--state-stride", str(STATE_STRIDE),
    ]
    # Omitted flags let the probe pick the task's own objects.
    if target:
        cmd += ["--target", target]
    if PLACEBO_TARGET.strip():
        cmd += ["--placebo-target", PLACEBO_TARGET.strip()]

log_path = out_dir / "probe.log"
print("$", " ".join(cmd), flush=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
# Colab's inline matplotlib backend is only valid inside the kernel process;
# inheriting it breaks LIBERO's import. The script guards this too.
env["MPLBACKEND"] = "Agg"
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"


def emit(chunk: bytes) -> None:
    """Write through to the cell output.

    Colab's sys.stdout is an ipykernel OutStream with no .buffer, so decode
    rather than assuming a binary stream exists.
    """
    stream = getattr(sys.stdout, "buffer", None)
    if stream is not None:
        stream.write(chunk)
    else:
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()


with log_path.open("wb") as log_handle:
    process = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    try:
        while True:
            chunk = process.stdout.read(4096)
            if not chunk:
                break
            log_handle.write(chunk); log_handle.flush()
            emit(chunk)
        rc = process.wait()
    except BaseException:
        # Never leave the probe running headless if the cell is interrupted.
        process.kill()
        process.wait()
        raise
if rc != 0:
    tail = log_path.read_text(errors="replace").splitlines()[-40:]
    print(f"\n--- last {len(tail)} log lines ({log_path}) ---", flush=True)
    for line in tail:
        print(line[-500:], flush=True)
    raise subprocess.CalledProcessError(rc, cmd)

if LIST_OBJECTS_ONLY:
    display(Markdown(
        "### Objects listed\n\nSet `LIST_OBJECTS_ONLY = False` to run the measurement. "
        "Override `TARGET` / `PLACEBO_TARGET` only if you do not want the task's own objects."
    ))
else:
    shapes_path = out_dir / "observation_shapes.json"
    if shapes_path.exists():
        shapes = json.loads(shapes_path.read_text())
        display(Markdown("### Observation Shapes Sent To The Policy"))
        display(HTML(
            "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
            + "".join(f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
                      for k, v in sorted(shapes.get("tensor_shapes", {}).items()))
            + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
              f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
        ))
        print("task:", repr(shapes.get("task")))

    summary_path = out_dir / "counterfactual_summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        display(Markdown("### Counterfactual Activation Response"))
        display(HTML(
            "<table><tr><th>state</th><th>kind</th><th>target</th><th>dose</th>"
            "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
            + "".join(
                "<tr><td>{s}</td><td>{k}</td><td>{t}</td><td>{d}</td><td>{px:.5f}</td>"
                "<td>{lat}</td><td>{act:.5g}</td></tr>".format(
                    s=r["state_index"], k=r["kind"], t=r["target"], d=r["dose"],
                    px=r["pixel"]["changed_pixel_fraction"],
                    lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                         else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                    act=r["action_relative_l2"])
                for r in summary.get("measurements", []))
            + "</table>"
        ))
        print("Read every row against the 'null' rows: that is the nondeterminism floor.")

    for png in sorted((out_dir / "images").glob("*.png"))[:12]:
        display(Markdown(f"**{png.name}**"))
        display(IPyImage(filename=str(png)))

    # Rank features by selectivity: responds to the task object, not to the
    # visually matched control.
    report_cmd = [str(PYTHON), "-u", "scripts/report_pi05_counterfactual_features.py",
                  str(out_dir), "--top", "30"]
    print("\n$", " ".join(report_cmd), flush=True)
    report = subprocess.run(report_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(report.stdout)
    if report.returncode != 0:
        print(report.stderr[-2000:])

    # Trace the circuit: which features feed which, named L<layer>/F<feature>.
    trace_cmd = [str(PYTHON), "-u", "scripts/trace_pi05_transcoder_circuit.py", str(out_dir),
                 "--checkpoint", TRANSCODER_CHECKPOINT, "--nodes-per-layer", "6", "--top-edges", "150"]
    print("\n$", " ".join(trace_cmd), flush=True)
    trace = subprocess.run(trace_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(trace.stdout[-3000:])
    if trace.returncode != 0:
        print(trace.stderr[-2000:])
    else:
        svg = out_dir / "circuit" / "circuit.svg"
        if svg.exists():
            display(Markdown("### Circuit Trace"))
            display(HTML(svg.read_text()))

print("\nArtifacts:", out_dir)
